# 목표

뉴스의 카테고리 예측

In [82]:
# %load_ext colablinter

# 데이터 파악

In [83]:
# 시각화, 디바이스 등 기본 설정

import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gc
import os


# 시각화 관련 설정
try:
    plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
except:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)


# # 로그
# import logging

# def init_logger() -> logging.Logger:
#     logging.basicConfig(
#         format="%(asctime)s [%(levelname)s] (%(filename)s:%(lineno)d) - %(message)s",
#         datefmt="%Y-%m-%d %H:%M:%S",
#         level=logging.INFO,
#         encoding="utf-8",
#     )
#     return logging.getLogger("")

# logger = init_logger()

In [84]:
ROOT_DIR = os.getcwd()
DATA_DIR = os.path.join(ROOT_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "20news-bydate-train")
TEST_DIR = os.path.join(DATA_DIR, "20news-bydate-test")

In [85]:
# 파일 개수 세기
from glob import glob

train_category_list = os.listdir(TRAIN_DIR)
test_category_list = os.listdir(TEST_DIR)

train_path_list = [path_ for path_ in glob(os.path.join(TRAIN_DIR, "**", "**")) 
               if os.path.isfile(path_)]
test_path_list = [path_ for path_ in glob(os.path.join(TEST_DIR, "**", "**")) 
              if os.path.isfile(path_)]

print(f"· 학습 데이터: {len(train_path_list)}개")
print(f"· 테스트 데이터: {len(test_path_list)}개")

· 학습 데이터: 11314개
· 테스트 데이터: 7532개


In [86]:
import chardet
from collections import Counter

def detect_encoding(file_path):
    """파일의 인코딩을 감지"""
    with open(file_path, 'rb') as f:
        raw_data = f.read()
    result = chardet.detect(raw_data)
    return result['encoding'], result['confidence']

# 모든 파일의 인코딩 정보 수집
encoding_info = {}
encoding_counter = Counter()

all_paths = train_path_list + test_path_list

for i, path in enumerate(all_paths):   
    try:
        encoding, confidence = detect_encoding(path)
        encoding_info[path] = {'encoding': encoding, 'confidence': confidence}
        encoding_counter[encoding] += 1
    except Exception as e:
        print(f"오류 발생 ({path}): {e}")

print("[인코딩 분포]")
for encoding, count in encoding_counter.most_common():
    percentage = (count / len(all_paths)) * 100
    print(f"· {encoding}: {count}개 ({percentage:.2f}%)")

[인코딩 분포]
· ascii: 18767개 (99.58%)
· ISO-8859-1: 58개 (0.31%)
· Windows-1252: 7개 (0.04%)
· MacRoman: 6개 (0.03%)
· None: 3개 (0.02%)
· HZ-GB-2312: 2개 (0.01%)
· Johab: 1개 (0.01%)
· ISO-2022-JP: 1개 (0.01%)
· ISO-8859-7: 1개 (0.01%)


In [87]:
def get_text(path):
    """감지된 인코딩으로 파일 읽기"""
    # encoding_info에 정보가 있으면 사용
    encoding = encoding_info.get(path, {}).get('encoding', 'latin-1')
    
    # None이면 latin-1 사용 (모든 바이트값을 읽을 수 있음)
    if encoding is None:
        encoding = 'latin-1'
    
    try:
        with open(path, "r", encoding=encoding) as f:
            return f.read()
    except:
        # 실패시 latin-1로 폴백 (절대 실패하지 않음)
        with open(path, "r", encoding="latin-1") as f:
            return f.read()

In [88]:
train_num_list = [path_.split("/")[-1] for path_ in train_path_list]
test_num_list = [path_.split("/")[-1] for path_ in test_path_list]

intersection = set(train_num_list) & set(test_num_list)

print(f"· 학습 - 테스트 데이터 겹치는 번호: {len(intersection)}개")

· 학습 - 테스트 데이터 겹치는 번호: 1294개


In [89]:
print("[겹치는 번호 오류 점검]")

train_dup_dict = {path_.split("/")[-1]: path_.split("/")[-2] for path_ in train_path_list 
               if path_.split("/")[-1] in intersection}

test_dup_dict = {path_.split("/")[-1]: path_.split("/")[-2] for path_ in test_path_list
               if path_.split("/")[-1] in intersection}


dup_path_list = []

for file_num in intersection:
    train_path = os.path.join(TRAIN_DIR, train_dup_dict[file_num], file_num)
    test_path = os.path.join(TEST_DIR, test_dup_dict[file_num], file_num)

    train_intersection = get_text(train_path).strip()
    test_intersection = get_text(test_path).strip()

    if train_intersection == test_intersection:
        dup_path_list.append(train_path)

print(f"· 내용 같은 파일: {len(dup_path_list)}개")

[겹치는 번호 오류 점검]
· 내용 같은 파일: 0개


단순히 번호만 겹쳤던 것으로 확인된다.

In [90]:
train_category_set = set(train_category_list)
test_category_set = set(test_category_list)

only_in_train = train_category_set - test_category_set
only_in_test = test_category_set - train_category_set

print("[카테고리 결손 파악]")
print(f"· 학습 데이터에만 있는 카테고리: {len(only_in_train)}개")
print(f"· 테스트 데이터에만 있는 카테고리: {len(only_in_test)}개")

[카테고리 결손 파악]
· 학습 데이터에만 있는 카테고리: 0개
· 테스트 데이터에만 있는 카테고리: 0개


In [91]:
CATEGORY_LIST = os.listdir(TRAIN_DIR)

In [92]:
print("[카테고리별 데이터 개수]")
i = 0

for category in CATEGORY_LIST:

    i += 1
    path_list = [path_ for path_ in glob(os.path.join(TRAIN_DIR, category, "*")) 
                 if os.path.isfile(path_)]
    
    print(f"{i}) {category}: {len(path_list)}개")

[카테고리별 데이터 개수]
1) talk.politics.mideast: 564개
2) rec.autos: 594개
3) comp.sys.mac.hardware: 578개
4) alt.atheism: 480개
5) rec.sport.baseball: 597개
6) comp.os.ms-windows.misc: 591개
7) rec.sport.hockey: 600개
8) sci.crypt: 595개
9) sci.med: 594개
10) talk.politics.misc: 465개
11) rec.motorcycles: 598개
12) comp.windows.x: 593개
13) comp.graphics: 584개
14) comp.sys.ibm.pc.hardware: 590개
15) sci.electronics: 591개
16) talk.politics.guns: 546개
17) sci.space: 593개
18) soc.religion.christian: 599개
19) misc.forsale: 585개
20) talk.religion.misc: 377개


- talk.religion.misc 카테고리가 비교적 적다.

- 상위 하위 카테고리로 구성되어 있으니, 카테고리도 더 나눠볼 필요가 있다.

- misc는 잡동사니라는 뜻으로, 주요 카테고리가 아니라는 뜻이다.

    - misc가 아닌 카테고리는 main으로 분류한다.

In [93]:
TRAIN_DICT = dict()

for category in CATEGORY_LIST:
    for path_ in glob(os.path.join(TRAIN_DIR, category, "*")):

        if os.path.isfile(path_):
            
            TRAIN_DICT[path_] = dict()
            TRAIN_DICT[path_]["category"] = category

            split_category = category.split(".")
            if split_category[-1] != "misc":
                split_category.append("main")

            for i, split in enumerate(split_category):
                if split == "main":
                    TRAIN_DICT[path_][5] = "main"
                    
                elif split == "misc" and i != 0:
                    TRAIN_DICT[path_][5] = "misc"

                else:
                    TRAIN_DICT[path_][i] = split

In [94]:
print("[TRAIN_DICT 예시 출력]")
for key, value in TRAIN_DICT.items():
    print(key, "\n: ", value)
    break

[TRAIN_DICT 예시 출력]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75895 
:  {'category': 'talk.politics.mideast', 0: 'talk', 1: 'politics', 2: 'mideast', 5: 'main'}


In [95]:
# import json

# train_json_path = os.path.join(DATA_DIR, "train.json")

# with open(train_json_path, "w", encoding="utf-8") as f:
#     json.dump(TRAIN_DICT, f, indent=4)


# test_json_path = os.path.join(DATA_DIR, "test.json")

# with open(test_json_path, "w", encoding="utf-8") as f:
#     json.dump(TEST_DICT, f, indent=4)

In [96]:
count_by_category_dict = {
              0: {},
              1: {},
              2: {},
              3: {},
              4: {},
              5: {}
              }


for category_dict in TRAIN_DICT.values():
    for key, value in category_dict.items():
        if key != "category":
            if value not in count_by_category_dict[key].keys():
                count_by_category_dict[key][value] = 1
            else:
                count_by_category_dict[key][value] += 1

In [97]:
import plotly.graph_objects as go

# 데이터
data = count_by_category_dict

nodes_by_level_and_name = {}
nodes = {}
edges = set()
node_id = 0

root_key = ('ROOT', 0)
nodes[root_key] = {'id': node_id, 'label': 'ROOT', 'level': 0, 'count': len(TRAIN_DICT)}
nodes_by_level_and_name[(0, 'ROOT')] = root_key
node_id += 1

for level, name_count_dict in data.items():
    current_level = level + 1
    for name, count in name_count_dict.items():
        node_key = (name, current_level)
        if node_key not in nodes:
            nodes[node_key] = {
                'id': node_id,
                'label': name,
                'level': current_level,
                'count': count,
            }
            nodes_by_level_and_name[(current_level, name)] = node_key
            node_id += 1

for _, category_dict in TRAIN_DICT.items():
    ordered_path = sorted(
        (int(k), v) for k, v in category_dict.items() if k != 'category'
    )
    parent_key = root_key
    for level_idx, name in ordered_path:
        current_level = level_idx + 1
        node_key = nodes_by_level_and_name.get((current_level, name))
        if node_key is None:
            continue
        edges.add((parent_key, node_key))
        parent_key = node_key

levels = {}
for node_key, node in nodes.items():
    levels.setdefault(node['level'], []).append(node_key)


# 노드의 최대 자식 깊이를 계산하는 함수
def get_max_depth(node_key, edges, memo=None):
    """
    해당 노드에서 시작하여 도달할 수 있는 최대 깊이를 반환합니다.
    """
    if memo is None:
        memo = {}
    
    if node_key in memo:
        return memo[node_key]
    
    # 자식 노드 찾기
    children = [child for parent, child in edges if parent == node_key]
    
    if not children:
        memo[node_key] = 0
        return 0
    
    # 자식들의 최대 깊이 + 1
    max_child_depth = max(get_max_depth(child, edges, memo) for child in children)
    memo[node_key] = max_child_depth + 1
    
    return memo[node_key]


# 부모-자식 관계를 기반으로 노드 정렬 함수
def sort_nodes_by_hierarchy(node_keys, edges, nodes):
    """
    부모-자식 관계를 고려하여 노드를 정렬합니다.
    깊이가 깊은 노드를 왼쪽에 배치하고 (레벨 5 제외),
    같은 부모를 가진 노드들끼리 그룹화합니다.
    """
    if not node_keys:
        return []
    
    # 각 노드의 부모들 찾기
    node_parents = {}
    for parent, child in edges:
        if child in node_keys:
            if child not in node_parents:
                node_parents[child] = []
            node_parents[child].append(parent)
    
    # 부모가 없는 노드들 (루트 레벨)
    if not node_parents:
        return sorted(node_keys, key=lambda x: nodes[x]['label'])
    
    # 각 노드의 최대 깊이 계산
    depth_memo = {}
    for node_key in node_keys:
        get_max_depth(node_key, edges, depth_memo)
    
    # 부모별로 자식 노드 그룹화
    parent_to_children = {}
    for node in node_keys:
        parents = tuple(sorted(node_parents.get(node, [])))
        if parents not in parent_to_children:
            parent_to_children[parents] = []
        parent_to_children[parents].append(node)
    
    # 각 그룹 내에서 정렬
    # 레벨 5가 아니면: 깊이가 깊은 것 -> 알파벳순
    # 레벨 5이면: 알파벳순만
    for parents in parent_to_children:
        children = parent_to_children[parents]
        current_level = nodes[children[0]]['level']
        
        if current_level == 6:  # 레벨 5 (0-indexed이므로 6)
            # 레벨 5는 알파벳순으로만 정렬
            parent_to_children[parents].sort(key=lambda x: nodes[x]['label'])
        else:
            # 다른 레벨은 깊이 우선 -> 알파벳순
            parent_to_children[parents].sort(
                key=lambda x: (-depth_memo.get(x, 0), nodes[x]['label'])
            )
    
    # 부모의 x 좌표 순서대로 자식들을 배치
    sorted_nodes = []
    
    # 첫 번째 레벨은 부모의 x 좌표가 없으므로 깊이 -> 알파벳순으로
    if all(p not in nodes or 'x' not in nodes[p] for parents in parent_to_children.keys() for p in parents):
        sorted_parent_groups = sorted(
            parent_to_children.keys(),
            key=lambda parents: (
                -max(depth_memo.get(child, 0) for child in parent_to_children[parents]),
                min(nodes[child]['label'] for child in parent_to_children[parents])
            )
        )
    else:
        # 부모의 x 좌표를 기준으로 정렬
        def get_parent_x(parents):
            if not parents:
                return 0
            parent_xs = [nodes[p].get('x', 0) for p in parents if p in nodes]
            return sum(parent_xs) / len(parent_xs) if parent_xs else 0
        
        sorted_parent_groups = sorted(parent_to_children.keys(), key=get_parent_x)
    
    for parents in sorted_parent_groups:
        sorted_nodes.extend(parent_to_children[parents])
    
    return sorted_nodes


# 레벨별 노드 배치 (계층 구조 기반 정렬 적용)
for level, node_keys in levels.items():
    # 계층 구조 기반 정렬 적용
    sorted_node_keys = sort_nodes_by_hierarchy(node_keys, edges, nodes)
    
    num_nodes = len(sorted_node_keys)
    y = -level * 2
    for i, key in enumerate(sorted_node_keys):
        if num_nodes == 1:
            x = 0
        else:
            spacing = min(15, 50 / num_nodes)
            x = (i - (num_nodes - 1) / 2) * spacing
        nodes[key]['x'] = x
        nodes[key]['y'] = y

edge_x = []
edge_y = []
for parent, child in edges:
    edge_x.extend([nodes[parent]['x'], nodes[child]['x'], None])
    edge_y.extend([nodes[parent]['y'], nodes[child]['y'], None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=1.5, color='#cccccc'),
    hoverinfo='none'
)

node_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
node_x = []
node_y = []
node_labels = []
node_counts = []
node_color = []
node_sizes = []

for n in nodes.values():
    node_x.append(n['x'])
    node_y.append(n['y'])
    node_labels.append(n['label'])
    node_counts.append(str(n.get('count', 0)))
    node_color.append(node_colors[n['level'] % len(node_colors)])
    base_size = 14 if n['level'] == 0 else 10
    scale = 0 if n.get('count') is None else max(0, n['count']) ** 0.5
    node_sizes.append(base_size + scale)

parent_counts = {}
for parent, child in edges:
    parent_counts[child] = parent_counts.get(child, 0) + 1

hover_text = []
for node_key, n in nodes.items():
    parent_count = parent_counts.get(node_key, 0)
    base_text = f"{n['label']}<br>레벨: {n['level']}<br>개수: {n.get('count', 0)}"
    if parent_count > 1:
        hover_text.append(base_text + f"<br>부모 노드: {parent_count}개")
    else:
        hover_text.append(base_text)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers',
    marker=dict(
        size=node_sizes,
        color=node_color,
        line=dict(width=2, color='white')
    ),
    hoverinfo='text',
    hovertext=hover_text
)

count_text_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='text',
    text=node_counts,
    textposition='middle center',
    textfont=dict(size=10, color='white', family='Arial'),
    hoverinfo='none'
)

label_text_trace = go.Scatter(
    x=node_x,
    y=[y + 0.5 for y in node_y],
    mode='text',
    text=node_labels,
    textposition='top center',
    textfont=dict(size=10, color='black', family='Arial'),
    hoverinfo='none'
)

layout = go.Layout(
    title=dict(text='카테고리 계층 구조 (중복 노드 병합)', font=dict(size=20)),
    showlegend=False,
    hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#ffffff',
    paper_bgcolor='#ffffff',
    margin=dict(l=40, r=40, t=60, b=40),
    height=900,
)

fig = go.Figure(
    data=[edge_trace, node_trace, count_text_trace, label_text_trace],
    layout=layout,
)
fig.show()


In [98]:
sub_category_list = list()
for category in CATEGORY_LIST:
    sub_category_list.extend(category.split(".")[1:])


multi_list = list()
for sub_category in set(sub_category_list):
    if sub_category_list.count(sub_category) > 1:
        multi_list.append(sub_category)


multi_dict = dict()
for category in CATEGORY_LIST:
    for multi_class in multi_list:
        if multi_class in category.split(".")[1:]:
            if multi_class in multi_dict.keys():
                multi_dict[multi_class].append(category)
            else:
                multi_dict[multi_class] = [category]


print("[여러 요소가 있는 카테고리]")
for key, value in multi_dict.items():
    print(f"· {key}: {value}")

[여러 요소가 있는 카테고리]
· politics: ['talk.politics.mideast', 'talk.politics.misc', 'talk.politics.guns']
· sys: ['comp.sys.mac.hardware', 'comp.sys.ibm.pc.hardware']
· hardware: ['comp.sys.mac.hardware', 'comp.sys.ibm.pc.hardware']
· sport: ['rec.sport.baseball', 'rec.sport.hockey']
· misc: ['comp.os.ms-windows.misc', 'talk.politics.misc', 'talk.religion.misc']
· religion: ['soc.religion.christian', 'talk.religion.misc']


1. 각 카테고리를 합쳤을 때는 불균형이 그렇게 심하게 보이지 않지만, 쪼개서 노드로 보니 결과가 다르다.

2. religion이라는 카테고리는 2개의 메인 카테고리에 딸려 있어 복잡하다. 계속 주시해야 하는 파트라는 생각이 든다.

3. 가장 하위 카테고리가 misc인 카테고리와 아닌 카테고리가 있는데 그것들만을 기준으로 보면 불균형은 아주 심하다.

4. hardware는 다른 깊이에 존재한다.

# 실험 계획

1. 주 카테고리부터 한 단계씩 밑으로 내려가며 학습. 각 단계마다 카테고리 불균형 해소 진행.

2. 주 카테고리와 가장 하위 카테고리(main, misc 제외) 학습

3. 그냥 한 번에 학습

카테고리 자체를 RNN처럼 시퀀셜로 학습할 수 있을까?

이 세 개로 나누어서 비교한다.

# 데이터 전처리 

## 1. 테스트 데이터 형식 조정

In [99]:
TEST_DICT = dict()

for category in CATEGORY_LIST:
    for path_ in glob(os.path.join(TEST_DIR, category, "*")):

        if os.path.isfile(path_):
            
            TEST_DICT[path_] = dict()
            TEST_DICT[path_]["category"] = category

            split_category = category.split(".")
            if split_category[-1] != "misc":
                split_category.append("main")

            for i in range(0, len(split_category)):
                TEST_DICT[path_][i] = split_category[i]


for _, dictionary in TEST_DICT.items():
    for i in range(1, len(dictionary) - 1):
        if dictionary[i] == "main":
            dictionary[5] = "main"
            del dictionary[i]
            continue

        elif dictionary[i] == "misc":
            dictionary[5] = "misc"
            del dictionary[i]

## 2. 텍스트 정제

In [100]:
import random

rand_idx = random.randint(1, len(TRAIN_DICT.keys()))
rand_key = list(TRAIN_DICT.keys())[rand_idx]

sample_key = rand_key.split("/")[-2]
sample_text = get_text(rand_key)

print(f"[샘플 출력]")
print(f"· 카테고리: {sample_key}")
print("· 본문", "="*40)
print(sample_text)

[샘플 출력]
· 카테고리: rec.motorcycles
· 본문 ========================================
From: asphaug@lpl.arizona.edu (Erik Asphaug x2773)
Subject: Re: CAMPING was Help with backpack
Organization: Lunar & Planetary Laboratory, Tucson AZ.
Lines: 24

In article <1993Apr14.193739.13359@rtsg.mot.com> svoboda@rtsg.mot.com (David Svoboda) writes:
>In article <1993Apr13.152706.27518@bnr.ca> Dave Dal Farra <gpz750@bnr.ca> writes:
>|My crafty girfriend makes campfire/bbq starters a la McGiver:
>Well, heck, if you're going to make them yourself, you can buy
>candle-wax by the pound--much cheper than the candles themselves.

Hell, just save your candle stubs and bring them.  Light them up, and
dribble the wax all over the kindling wood and light _that_.  Although
I like the belly-button lint / eggshell case idea the best, if you're
feeling particularly industrious some eventful evening.  Or you can
do what I did one soggy summer: open the fuel line, drain some onto a 
piece of rough or rotten wood, stick t

### 2-1. header, footer, 메타데이터

인용 부호는 아래의 답장 구조에서 활용하므로

현 파트에서는 제거하지 않는다.

In [101]:
import re

def preprocess_text(text):
    """
    뉴스그룹 메타데이터 제거 함수
    (인용 제거는 remove_quoted_content()에서 처리)
    """
    
    # 0. 짧은 줄 병합: 글자 수 초과로 인용 부호 없이 아래로 내려간 글자 병합
    lines = text.split('\n')
    merged_lines = []
    
    for i, line in enumerate(lines):
        stripped = line.strip()
        
        # 빈 줄 보존
        if not stripped:
            merged_lines.append(line)
            continue
        
        # 매우 짧고, 이전 줄이 마침표로 끝나지 않으면 병합
        if (len(stripped) < 10 and 
            merged_lines and 
            merged_lines[-1].strip() and
            not re.search(r'[.!?]$', merged_lines[-1])):
            merged_lines[-1] += ' ' + stripped
        else:
            merged_lines.append(line)
    
    text = '\n'.join(merged_lines)


    # 1. "In article ... writes:" 패턴 제거
    text = re.sub(r'In (article|message) <[^>]+>.*?writes?:', '', text, flags=re.IGNORECASE)
    
    # 2. "On ... wrote:" 패턴 제거
    text = re.sub(r'On .+? wrote:', '', text, flags=re.IGNORECASE)
    
    # 3. 이메일 주소 + writes 패턴 제거
    text = re.sub(r'\S+@\S+.*?writes?:', '', text, flags=re.IGNORECASE)
    
    # 4. 헤더 필드 제거
    header_fields = [
        'Lines:', 'From:', 'Organization:', 'Article-I.D.:',
        'Newsgroups:', 'Path:', 'Sender:', 'Nntp-Posting-Host:'
        'Distribution:', 'Message-ID:', 'References:',
        'NNTP-Posting-Host:', 'Date:', 'Reply-To:',
        'Keywords:', 'Summary:', 'Expires:', 'Followup-To:',
        'X-Newsreader:', 'X-Mailer:', 'Posted:', 'Xref:', 'X-UserAgent:',
        'Nntp-Posting-Host:', 'Distribution:', 'Originator:'
    ]
    for field in header_fields:
        text = re.sub(rf'^.*{re.escape(field)}.*$', '', text, flags=re.MULTILINE)
    
    # 5. Article ID 패턴 제거
    text = re.sub(r'<[^>]+@[^>]+>', '', text)
    
    # 6. 서명 블록 제거 (-- 이후)
    lines = text.split('\n')
    footer_start = len(lines)
    
    # 서명 구분선 (-- 또는 ---) 찾기
    for i in range(len(lines) - 1, -1, -1):
        stripped = lines[i].strip()
        if re.match(r'^--+\s*$', stripped) or stripped == '-- ':
            footer_start = i
            break
    
    # 구분선 없으면 역방향 스캔 (빈 줄 기반)
    if footer_start == len(lines):
        found_footer_hint = False
        
        for i in range(len(lines) - 1, -1, -1):
            stripped = lines[i].strip()
            
            # 빈 줄 만나면 중단
            if not stripped:
                if found_footer_hint:
                    break
                continue
            
            # footer 패턴 체크 (힌트)
            is_footer_hint = (
                re.match(r'^[\w\.\-]+@[\w\.\-]+', stripped) or  # 이메일
                re.search(r'(University|College|Department|Institute|Corp\.|Inc\.|Ltd\.)', stripped, re.IGNORECASE) or
                re.search(r'\d{3}[-.\s]\d{3}[-.\s]\d{4}', stripped) or  # 전화번호
                'PGP' in stripped.upper()
            )
            
            if is_footer_hint:
                found_footer_hint = True
                footer_start = i
    
    # footer 제거
    if footer_start < len(lines):
        text = '\n'.join(lines[:footer_start])
    
    # 7. 특수 문자 경계선 제거
    text = re.sub(r'^[-=*]{3,}$', '', text, flags=re.MULTILINE)
    
    # 8. 연속 빈 줄 정리
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # 9. 같은 줄 내 다중 공백 정리
    text = re.sub(r'[ \t]+', ' ', text)
    
    # 10. 앞뒤 공백 제거
    text = text.strip()
    
    return text

print(preprocess_text(sample_text))

Subject: Re: CAMPING was Help with backpack

>
>|My crafty girfriend makes campfire/bbq starters a la McGiver:
>Well, heck, if you're going to make them yourself, you can buy
>candle-wax by the pound--much cheper than the candles themselves.

Hell, just save your candle stubs and bring them. Light them up, and
dribble the wax all over the kindling wood and light _that_. Although
I like the belly-button lint / eggshell case idea the best, if you're
feeling particularly industrious some eventful evening. Or you can
do what I did one soggy summer: open the fuel line, drain some onto a 
piece of rough or rotten wood, stick that into the middle of the soon-to-
be inferno and CAREFULLY strike a match... As Kurt Vonnegut titled one
of the latter chapters in Cat's Cradle, "Ah-Whoom!"

Works like a charm every time :-)

/-----b-o-d-y---i-s---t-h-e---b-i-k-e----------------------------\
| |
| DoD# 88888 asphaug@hindmost.lpl.arizona.edu |
| '90 Kawi Zephyr (Erik Asphaug) |
| '86 BMW R80GS |
\----

### 2-2. 답장 구조

답장 내용이 카테고리와 관련 없는 경우와 관련 있는 경우 모두 존재한다.

어떤 방향이든, 처음 쓰였던 내용이 계속 인용되어 여러 차례 학습에 노출된다는 점을 해결해야 한다.

1. ```writes: wrote:```로 답장 텍스트 여부 식별

2. 그 다음 문장부터 인용 부호가 담긴 문장 식별 -> `reply_dict.json` 생성

3. ```reply_dict``` 이용하여 원본 식별

4. 관계 분석하여 중복 내용 제거

In [102]:
len(new_list)

5340

In [103]:
# # reply_dict.json 최초 생성 코드
# import re
# import json

# quote_head = r'.+?\s+(writes|wrote):\s*'
# # ## TODO: In-Reply-To:
# # ## TODO: As quoted from + 이메일
# # ## TODO: Subject: Re:
# # ## TODO: 이중 답장: quote_mark 반복 횟수로 판단. relation_dict를 0: 1:로 계층화 할 수 있을 것 같다.
# # ## TODO: 누락됐던 TRAIN_DICT -> reply_dict.json 업데이트 해야 함.

# reply_dict_path = os.path.join(DATA_DIR, "reply_dict.json")
# with open(reply_dict_path, "r", encoding="utf-8") as f:
#     reply_dict = json.load(f)

# for text_path in TRAIN_DICT.keys():

#     text = get_text(text_path)

#     quote_index_list = [match.start() for match in re.finditer(quote_head, text)] # writes: 등의 인용 표현 탐지

#     e_idx = 0

#     if len(quote_index_list) > 0:
#         enter_index_list = [match.start() for match in re.finditer("\n", text)] # 줄바꿈별 분리
        
#         reply_dict[text_path] = list()

#         for quote_index in quote_index_list:
#             # e_dix < q_idx < e_dix + 1 설정하기
#             while enter_index_list[e_idx + 1] <= quote_index:
#                 e_idx += 1

#             # quote_sentence 찾기
#             q_idx = e_idx + 1
#             quote_sentence = text[enter_index_list[q_idx] : enter_index_list[q_idx + 1]].strip("\n")

#             if re.search(r"\w", quote_sentence):
#                 quote_quote_mark = re.search(r"^\W*", quote_sentence).group().strip("\n").strip()

#             else:
#                 q_idx += 1
#                 quote_sentence = text[enter_index_list[q_idx] : enter_index_list[q_idx + 1]].strip("\n")
#                 quote_quote_mark = re.search(r"^\W*", quote_sentence).group().strip("\n").strip()


#             reply_dict[text_path].append({"quote_mark": quote_quote_mark,
#                                           "quote_sentence": quote_sentence})

In [104]:
import json

def open_reply_dict():

    reply_dict_path = os.path.join(DATA_DIR, "reply_dict.json")

    with open(reply_dict_path, "r", encoding="utf-8") as f:
        reply_dict = json.load(f)
    
    return reply_dict

reply_dict = open_reply_dict()

In [105]:
# 인용 부호와 혼동될 수 있는 기호들 추적 -> reply_dict.json 직접 수정 진행

yoju_set = {"/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/comp.sys.mac.hardware/50457",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/54180",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/176895",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/178311",
             "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/83689"}

def list_quote_endswith(text: str):
    for text_path, quote_dict_list in reply_dict.items():
        if text_path not in yoju_set:
            quote_mark_list = list([quote_dict["quote_mark"], quote_dict["quote_sentence"]] 
                                   for quote_dict in quote_dict_list 
                                   if quote_dict["quote_mark"].endswith(text))

            if quote_mark_list != []:
                print(text_path)
                print(quote_mark_list)

In [106]:
list_quote_endswith("[")

yoju_set.update(["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/51318",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/82795",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75406",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/176993",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/soc.religion.christian/20709"
                 ])

/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75406
[['[', '[ stuff deleted ]']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.misc/176993
[['>[', '    >[START OF DOCUMENT: doclist.txt.lis ]']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.space/61135
[['[', '[ a nearly perfect parody  -- needed more random CAPS]']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/comp.windows.x/67253
[['[', '       [Hint for Sun OS users:  use /usr/5bin/echo instead of']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/soc.religion.christian/20709
[['[', '   [There may be some misunderstanding over terms here...]']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/soc.religion.christian/21374
[['[', '[why are atheists atheists/ believes it could be the result of']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-

In [107]:
list_quote_endswith("(")

yoju_set.update(["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/alt.atheism/51318",
                 "/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.religion.misc/82795",
                 ])

In [108]:
list_quote_endswith("<")

yoju_set.update(["/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/rec.autos/101572",
                 ])

/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/75394
[['><', '><disclaimer: If there is anybody on USENET dumb enough to interpret']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/rec.autos/101572
[['<', "<apparently you're not a woman - my husband hates the auto door locks"]]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/rec.autos/101674
[['> <', "> <apparently you're not a woman - my husband hates the auto door locks"]]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15589
[['<', '<If Clipper comes to cellular phones along with legal proscriptions against']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15667
[['<', '<the case that all the Republicans, etc. in the NSA and FBI and CIA']]
/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15555
[['<', '<As usually, you are not reading. The propo

In [109]:
list_quote_endswith("{")

In [110]:
reply_dict = open_reply_dict()


rand_idx = random.randint(0, len(reply_dict) - 1)
rand_key = list(reply_dict.keys())[rand_idx]

print("[reply_dict.json]")
print(f"· 답장 데이터: {len(reply_dict)}개")
print(f"· 샘플 key: {rand_key}")
print(f"· 샘플 value: {reply_dict[rand_key]}")

[reply_dict.json]
· 답장 데이터: 5975개
· 샘플 key: /Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/sci.crypt/15000
· 샘플 value: [{'quote_mark': ':', 'quote_sentence': ':As a matter of fact, i do keep random files on my disk.  The reason is,'}]


# 데이터셋 생성

1. 원본 - 답장 관계를 파악하여 중복되는 데이터 계층 파악

2. 글을 단위로 쪼개어 별도의 데이터셋 생성

In [111]:
from collections import defaultdict
from typing import Dict, List, Set, Tuple
import hashlib

# ==================== 1단계: 원본-답장 관계 그래프 구축 ====================

def build_relationship_graph(reply_dict, TRAIN_DIR):
    """
    답장 파일들의 원본을 찾아 관계 그래프 구축
    답장끼리의 관계도 파악 (답장의 답장)
    """
    reply_to_origins = defaultdict(list)
    origin_to_replies = defaultdict(list)
    reply_to_replies = defaultdict(list)
    no_origin_replies = set()
    
    all_reply_paths = set(reply_dict.keys())
    
    for reply_path, quote_dict_list in reply_dict.items():
        category = reply_path.split("/")[-2]
        
        # 후보: 같은 카테고리의 모든 파일 (답장 포함)
        candidate_path_list = [path_ for path_ in glob(os.path.join(TRAIN_DIR, category, "**")) 
                               if path_ != reply_path and os.path.isfile(path_)
                               ]
        
        # 인용문 전처리
        raw_quote_list = []
        processed_quote_list = []
        
        for quote_dict in quote_dict_list:
            quote_sentence = quote_dict["quote_sentence"]
            quote_mark = quote_dict["quote_mark"]
            
            raw_quote_list.append(quote_sentence)
            
            if quote_mark:
                processed_quote_list.append(quote_sentence.split(quote_mark)[-1].strip())
            else:
                processed_quote_list.append(quote_sentence.strip())
        
        # 원본/부모 답장 찾기
        found_parent = False
        
        for candidate_path in candidate_path_list:
            candidate_text = get_text(candidate_path)
            
            # 첫 번째 인용문이 없으면 스킵
            if processed_quote_list[0] not in candidate_text:
                continue
            
            # 모든 인용문이 있는지 확인
            all_quotes_found = all(processed_quote in candidate_text 
                                   for processed_quote in processed_quote_list
                                   )
            
            if not all_quotes_found:
                continue
            
            # 인용 마크가 있는지 확인 (원본 vs 답장 구분)
            has_quote_marks = False
            for splitted_line in candidate_text.split("\n"):
                for raw_quote in raw_quote_list:
                    if re.search(rf'^\s*{re.escape(raw_quote)}', splitted_line):
                        has_quote_marks = True
                        break
                if has_quote_marks:
                    break
            
            if not has_quote_marks:
                # 인용 마크 없음  ->  원본
                if candidate_path not in all_reply_paths:
                    reply_to_origins[reply_path].append(candidate_path)
                    origin_to_replies[candidate_path].append(reply_path)
                    found_parent = True
                else:
                    # 인용 마크 없는 답장  ->  원본이 유실된 답장
                    reply_to_replies[reply_path].append(candidate_path)
                    found_parent = True
            else:
                # 인용 마크 있음  ->  다른 답장
                if candidate_path in all_reply_paths:
                    reply_to_replies[reply_path].append(candidate_path)
                    found_parent = True
        
        if not found_parent:
            no_origin_replies.add(reply_path)
    
    return reply_to_origins, origin_to_replies, reply_to_replies, no_origin_replies

In [112]:
# ==================== 2단계: 인용 부분 제거 ====================

def remove_quoted_content(text: str, quote_dict_list: List[Dict]) -> str:
    """
    인용 파트 제거(해당 파일에 특정된 인용 마크만 사용)
    """
    lines = text.split("\n")
    filtered_lines = []
    
    # 이 파일의 인용 마크만 수집
    quote_marks = set()
    for quote_dict in quote_dict_list:
        if quote_dict["quote_mark"]:
            quote_marks.add(quote_dict["quote_mark"])
    
    if not quote_marks:
        # 인용 마크가 없으면 원본 그대로 반환
        return text
    
    for line in lines:
        stripped_line = line.lstrip()
        
        # 인용 라인인지 확인
        is_quoted = False
        for mark in quote_marks:
            if stripped_line.startswith(mark):
                is_quoted = True
                break
        
        if not is_quoted:
            filtered_lines.append(line)
    
    return "\n".join(filtered_lines)

In [113]:
# ==================== 3단계: 문장/단락 단위로 분리 ====================

def split_into_chunks(text: str, min_length: int = 100) -> List[str]:
    """
    텍스트를 문장/단락 단위로 분리
    
    Args:
        text: 원본 텍스트
        min_length: 최소 청크 길이
    
    Returns:
        청크 리스트
    """
    # 빈 줄로 단락 분리
    paragraphs = text.split("\n\n")
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        
        # 현재 청크에 추가
        if current_chunk:
            current_chunk += "\n\n" + para
        else:
            current_chunk = para
        
        # 최소 길이 이상이면 청크 완성
        if len(current_chunk) >= min_length:
            chunks.append(current_chunk)
            current_chunk = ""
    
    # 남은 청크
    if current_chunk and len(current_chunk) >= min_length:
        chunks.append(current_chunk)
    
    return chunks

In [114]:
# ==================== 4단계: 중복 없는 데이터셋 구축 ====================

def build_deduplicated_dataset(reply_dict, train_path_list, TRAIN_DIR):
    """
    중복 없이 각 내용이 정확히 한 번씩만 포함되는 데이터셋 구축
    """
    print("<<데이터셋 구축>>")
    print()
    
    # 1) 관계 그래프 구축
    print("[원본-답장 관계 분석]")
    reply_to_origins, origin_to_replies, reply_to_replies, no_origin_replies = \
        build_relationship_graph(reply_dict, TRAIN_DIR)
    
    print(f"· 원본: {len(origin_to_replies)}개")
    print(f"· 원본-답장: {len(reply_to_origins)}개")
    print(f"· 답장-답장: {len(reply_to_replies)}개")
    print(f"· 원본 없는 답장: {len(no_origin_replies)}개")
    print()
    
    # 2) 각 파일별로 처리할 내용 결정
    print("[파일 구성]")
    
    file_processing = {}  # {경로: {'type': 'full'/'filtered', 'quote_info': ...}}
    processed_files = set()
    
    # 2-1) 원본 파일들: 전체 사용
    for origin_path in origin_to_replies.keys():
        file_processing[origin_path] = {
            'type': 'full',
            'quote_info': None
        }
        processed_files.add(origin_path)
    
    # 2-2) 원본이 있는 답장들: 인용 제거
    for reply_path, origin_paths in reply_to_origins.items():
        if origin_paths:  # 원본이 있으면
            file_processing[reply_path] = {
                'type': 'filtered',
                'quote_info': reply_dict[reply_path]
            }
            processed_files.add(reply_path)
    
    # 2-3) 원본 없는 답장: 전체 사용
    for no_origin_path in no_origin_replies:
        file_processing[no_origin_path] = {
            'type': 'full',
            'quote_info': None
        }
        processed_files.add(no_origin_path)
    
    # 2-4) 답장의 답장 관계 처리
    for reply_path, parent_replies in reply_to_replies.items():
        if reply_path not in file_processing:
            file_processing[reply_path] = {
                'type': 'filtered',
                'quote_info': reply_dict[reply_path]
            }
            processed_files.add(reply_path)
    
    # 2-5) 답장이 아닌 일반 파일들: 전체 사용
    for path in train_path_list:
        if path not in processed_files:
            file_processing[path] = {
                'type': 'full',
                'quote_info': None
            }
    
    print(f"· 전체 사용: {len([f for f in file_processing.values() if f['type'] == 'full'])}개")
    print(f"· 인용 제거: {len([f for f in file_processing.values() if f['type'] == 'filtered'])}개")
    print()
    
    # 3) 파일별로 텍스트 추출 및 청크 생성
    print("[텍스트 추출 및 분리]")
    
    all_chunks = []
    chunk_hashes = set()  # 중복 체크용
    
    for file_path, processing_info in file_processing.items():
        try:
            # 카테고리 추출
            category = file_path.split("/")[-2]

            # 세부 카테고리 추출
            split_category = dict()
            split_list = category.split(".")

            if split_list[-1] != "misc":
                split_list.append("main")

            for i, split in enumerate(split_list):
                if split == "main":
                    split_category[5] = "main"
                
                elif i > 1 and split == "misc":
                    split_category[5] = "misc"

                else:
                    split_category[i] = split

            # 텍스트 읽기
            text = get_text(file_path)
            text = preprocess_text(text)
            
            # 필터링 필요 시 인용 제거
            if processing_info['type'] == 'filtered':
                text = remove_quoted_content(text, processing_info['quote_info'])
            
            # 청크로 분리
            chunks = split_into_chunks(text, min_length=100)
            
            # 중복 제거하면서 추가
            for chunk in chunks:
                chunk_normalized = re.sub(r'\s+', ' ', chunk.strip())
                chunk_hash = hashlib.md5(chunk_normalized.encode('utf-8')).hexdigest()
                
                if chunk_hash not in chunk_hashes:
                    all_chunks.append({
                        'text': chunk.strip(),
                        'category': category,
                        'split_category': split_category,
                        'source': file_path,
                        'hash': chunk_hash
                    })
                    chunk_hashes.add(chunk_hash)
        
        except Exception as e:
            print(f"· 파일 처리 실패: {file_path}")
            print(f"· 에러 내용: {e}")
    
    print(f"· 총 데이터: {len(all_chunks)}개 (중복 제거 완료)")
    print()
    
    return all_chunks, file_processing

In [115]:
# ==================== 5단계: 중복 검증 ====================

def normalize_text(text: str) -> str:
    """텍스트 정규화"""
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


def verify_no_duplicates(dataset: List[Dict]) -> bool:
    """
    데이터셋에 중복이 없는지 검증
    """
    print("[중복 검증]")
    
    seen_hashes = set()
    duplicates = []
    
    for idx, item in enumerate(dataset):
        item_hash = item['hash']
        if item_hash in seen_hashes:
            duplicates.append(idx)
        else:
            seen_hashes.add(item_hash)
    

    print(f"· 중복 데이터: {len(duplicates)}개")

In [116]:
# 데이터셋 생성
final_dataset, file_processing = build_deduplicated_dataset(reply_dict, 
                                                            train_path_list,
                                                            TRAIN_DIR
                                                            )

# 중복 검증
is_clean = verify_no_duplicates(final_dataset)

<<데이터셋 구축>>

[원본-답장 관계 분석]
· 원본: 5328개
· 원본-답장: 1920개
· 답장-답장: 3839개
· 원본 없는 답장: 1427개

[파일 구성]
· 전체 사용: 6766개
· 인용 제거: 4548개

[텍스트 추출 및 분리]
· 총 데이터: 38579개 (중복 제거 완료)

[중복 검증]
· 중복 데이터: 0개


In [117]:
quote_dict_list = [b["quote_info"] for a, b in file_processing.items() if b["quote_info"] != None][0]

quote_marks = set()
for quote_dict in quote_dict_list:
    if quote_dict["quote_mark"]:
        quote_marks.add(quote_dict["quote_mark"])

quote_dict_list

[{'quote_mark': '>',
  'quote_sentence': '>WHERE CAN I JOIN THE SERDAR ARGIC FAN CLUB?  DO I GET A T-SHIRT?'}]

In [118]:
print(f"· 최종 데이터: {len(final_dataset)}개")

· 최종 데이터: 38579개


In [119]:
final_dataset[0]

{'text': 'Subject: To be exact, 2.5 million readers enlightened by Serdar Argic\n\n(a.k.a. Serdar Argic, The Merciful and Compassionate)',
 'category': 'talk.politics.mideast',
 'split_category': {0: 'talk', 1: 'politics', 2: 'mideast', 5: 'main'},
 'source': '/Users/won/dev/00_codeit/0_mission/10_NLP_RNN/data/20news-bydate-train/talk.politics.mideast/76210',
 'hash': 'b51150ccda00dde2560576ce91a7937e'}

In [120]:
# 결과 샘플 출력
rand_idx = random.randint(0, len(final_dataset) - 1)

print(f"[샘플 출력] {final_dataset[rand_idx]['source'].split('/')[-1]} | {final_dataset[rand_idx]['category']}")
print()
print(final_dataset[rand_idx]['text'])

## TODO: 인용 마크 안 없어지는 거 수정

[샘플 출력] 51691 | comp.sys.mac.hardware

Stuff that can go with it......
I've got 3 modems and I'd be willing to give 1 of the 9600's and the
2400 with the system


In [121]:
category_counts = defaultdict(int)
for chunk in final_dataset:
    category_counts[chunk['category']] += 1

print(f"총 카테고리 수: {len(category_counts)}개")
for category, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    percentage = count / len(final_dataset) * 100
    print(f"  · {category:25s}: {count:d}개 ({percentage:.1f}%)")

총 카테고리 수: 20개
  · talk.politics.mideast    : 2816개 (7.3%)
  · sci.crypt                : 2563개 (6.6%)
  · comp.windows.x           : 2532개 (6.6%)
  · soc.religion.christian   : 2527개 (6.6%)
  · sci.space                : 2305개 (6.0%)
  · talk.politics.guns       : 2263개 (5.9%)
  · rec.sport.hockey         : 2113개 (5.5%)
  · comp.graphics            : 2110개 (5.5%)
  · alt.atheism              : 2033개 (5.3%)
  · sci.med                  : 1980개 (5.1%)
  · talk.politics.misc       : 1751개 (4.5%)
  · comp.sys.ibm.pc.hardware : 1667개 (4.3%)
  · talk.religion.misc       : 1615개 (4.2%)
  · sci.electronics          : 1593개 (4.1%)
  · rec.sport.baseball       : 1587개 (4.1%)
  · rec.autos                : 1583개 (4.1%)
  · rec.motorcycles          : 1533개 (4.0%)
  · misc.forsale             : 1366개 (3.5%)
  · comp.os.ms-windows.misc  : 1345개 (3.5%)
  · comp.sys.mac.hardware    : 1297개 (3.4%)


In [ ]:
# 세부 카테고리 통계 및 시각화 준비
count_by_category_dict = defaultdict(lambda: defaultdict(int))

for chunk in final_dataset:
    split_cat = chunk['split_category']
    
    # 각 레벨별로 카운트
    for level, name in split_cat.items():
        count_by_category_dict[level][name] += 1

In [123]:
data = count_by_category_dict

nodes_by_level_and_name = {}
nodes = {}
edges = set()
node_id = 0

root_key = ('ROOT', 0)
nodes[root_key] = {'id': node_id, 'label': 'ROOT', 'level': 0, 'count': len(final_dataset)}
nodes_by_level_and_name[(0, 'ROOT')] = root_key
node_id += 1

for level, name_count_dict in data.items():
    current_level = level + 1
    for name, count in name_count_dict.items():
        node_key = (name, current_level)
        if node_key not in nodes:
            nodes[node_key] = {
                'id': node_id,
                'label': name,
                'level': current_level,
                'count': count,
            }
            nodes_by_level_and_name[(current_level, name)] = node_key
            node_id += 1

for _, category_dict in TRAIN_DICT.items():
    ordered_path = sorted(
        (int(k), v) for k, v in category_dict.items() if k != 'category'
    )
    parent_key = root_key
    for level_idx, name in ordered_path:
        current_level = level_idx + 1
        node_key = nodes_by_level_and_name.get((current_level, name))
        if node_key is None:
            continue
        edges.add((parent_key, node_key))
        parent_key = node_key

levels = {}
for node_key, node in nodes.items():
    levels.setdefault(node['level'], []).append(node_key)


# 노드의 최대 자식 깊이를 계산하는 함수
def get_max_depth(node_key, edges, memo=None):
    """
    해당 노드에서 시작하여 도달할 수 있는 최대 깊이를 반환합니다.
    """
    if memo is None:
        memo = {}
    
    if node_key in memo:
        return memo[node_key]
    
    # 자식 노드 찾기
    children = [child for parent, child in edges if parent == node_key]
    
    if not children:
        memo[node_key] = 0
        return 0
    
    # 자식들의 최대 깊이 + 1
    max_child_depth = max(get_max_depth(child, edges, memo) for child in children)
    memo[node_key] = max_child_depth + 1
    
    return memo[node_key]


# 부모-자식 관계를 기반으로 노드 정렬 함수
def sort_nodes_by_hierarchy(node_keys, edges, nodes):
    """
    부모-자식 관계를 고려하여 노드를 정렬합니다.
    깊이가 깊은 노드를 왼쪽에 배치하고 (레벨 5 제외),
    같은 부모를 가진 노드들끼리 그룹화합니다.
    """
    if not node_keys:
        return []
    
    # 각 노드의 부모들 찾기
    node_parents = {}
    for parent, child in edges:
        if child in node_keys:
            if child not in node_parents:
                node_parents[child] = []
            node_parents[child].append(parent)
    
    # 부모가 없는 노드들 (루트 레벨)
    if not node_parents:
        return sorted(node_keys, key=lambda x: nodes[x]['label'])
    
    # 각 노드의 최대 깊이 계산
    depth_memo = {}
    for node_key in node_keys:
        get_max_depth(node_key, edges, depth_memo)
    
    # 부모별로 자식 노드 그룹화
    parent_to_children = {}
    for node in node_keys:
        parents = tuple(sorted(node_parents.get(node, [])))
        if parents not in parent_to_children:
            parent_to_children[parents] = []
        parent_to_children[parents].append(node)
    
    # 각 그룹 내에서 정렬
    # 레벨 5가 아니면: 깊이가 깊은 것 -> 알파벳순
    # 레벨 5이면: 알파벳순만
    for parents in parent_to_children:
        children = parent_to_children[parents]
        current_level = nodes[children[0]]['level']
        
        if current_level == 6:  # 레벨 5 (0-indexed이므로 6)
            # 레벨 5는 알파벳순으로만 정렬
            parent_to_children[parents].sort(key=lambda x: nodes[x]['label'])
        else:
            # 다른 레벨은 깊이 우선 -> 알파벳순
            parent_to_children[parents].sort(
                key=lambda x: (-depth_memo.get(x, 0), nodes[x]['label'])
            )
    
    # 부모의 x 좌표 순서대로 자식들을 배치
    sorted_nodes = []
    
    # 첫 번째 레벨은 부모의 x 좌표가 없으므로 깊이 -> 알파벳순으로
    if all(p not in nodes or 'x' not in nodes[p] for parents in parent_to_children.keys() for p in parents):
        sorted_parent_groups = sorted(
            parent_to_children.keys(),
            key=lambda parents: (
                -max(depth_memo.get(child, 0) for child in parent_to_children[parents]),
                min(nodes[child]['label'] for child in parent_to_children[parents])
            )
        )
    else:
        # 부모의 x 좌표를 기준으로 정렬
        def get_parent_x(parents):
            if not parents:
                return 0
            parent_xs = [nodes[p].get('x', 0) for p in parents if p in nodes]
            return sum(parent_xs) / len(parent_xs) if parent_xs else 0
        
        sorted_parent_groups = sorted(parent_to_children.keys(), key=get_parent_x)
    
    for parents in sorted_parent_groups:
        sorted_nodes.extend(parent_to_children[parents])
    
    return sorted_nodes


# 레벨별 노드 배치 (계층 구조 기반 정렬 적용)
for level, node_keys in levels.items():
    # 계층 구조 기반 정렬 적용
    sorted_node_keys = sort_nodes_by_hierarchy(node_keys, edges, nodes)
    
    num_nodes = len(sorted_node_keys)
    y = -level * 2
    for i, key in enumerate(sorted_node_keys):
        if num_nodes == 1:
            x = 0
        else:
            spacing = min(15, 50 / num_nodes)
            x = (i - (num_nodes - 1) / 2) * spacing
        nodes[key]['x'] = x
        nodes[key]['y'] = y

edge_x = []
edge_y = []
for parent, child in edges:
    edge_x.extend([nodes[parent]['x'], nodes[child]['x'], None])
    edge_y.extend([nodes[parent]['y'], nodes[child]['y'], None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=1.5, color='#cccccc'),
    hoverinfo='none'
)

node_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
node_x = []
node_y = []
node_labels = []
node_counts = []
node_color = []
node_sizes = []

for n in nodes.values():
    node_x.append(n['x'])
    node_y.append(n['y'])
    node_labels.append(n['label'])
    node_counts.append(str(n.get('count', 0)))
    node_color.append(node_colors[n['level'] % len(node_colors)])
    base_size = 14 if n['level'] == 0 else 10
    scale = 0 if n.get('count') is None else max(0, n['count']) ** 0.5
    node_sizes.append(base_size + scale)

parent_counts = {}
for parent, child in edges:
    parent_counts[child] = parent_counts.get(child, 0) + 1

hover_text = []
for node_key, n in nodes.items():
    parent_count = parent_counts.get(node_key, 0)
    base_text = f"{n['label']}<br>레벨: {n['level']}<br>개수: {n.get('count', 0)}"
    if parent_count > 1:
        hover_text.append(base_text + f"<br>부모 노드: {parent_count}개")
    else:
        hover_text.append(base_text)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers',
    marker=dict(
        size=node_sizes,
        color=node_color,
        line=dict(width=2, color='white')
    ),
    hoverinfo='text',
    hovertext=hover_text
)

count_text_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='text',
    text=node_counts,
    textposition='middle center',
    textfont=dict(size=10, color='white', family='Arial'),
    hoverinfo='none'
)

label_text_trace = go.Scatter(
    x=node_x,
    y=[y + 0.5 for y in node_y],
    mode='text',
    text=node_labels,
    textposition='top center',
    textfont=dict(size=10, color='black', family='Arial'),
    hoverinfo='none'
)

layout = go.Layout(
    title=dict(text='카테고리 계층 구조 (중복 노드 병합)', font=dict(size=20)),
    showlegend=False,
    hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#ffffff',
    paper_bgcolor='#ffffff',
    margin=dict(l=40, r=40, t=60, b=40),
    height=900,
)

fig = go.Figure(
    data=[edge_trace, node_trace, count_text_trace, label_text_trace],
    layout=layout,
)
fig.show()


In [196]:
from torch.utils.data import Dataset

whole_class_map_by_cat = dict()
for i, category in enumerate(CATEGORY_LIST):
    whole_class_map_by_cat[category] = i

whole_class_map_by_num = dict()
for i, category in enumerate(CATEGORY_LIST):
    whole_class_map_by_num[i] = category


class TextDatasetByWhole(Dataset):
    def __init__(self):
        self.list = final_dataset
        self.class_map = whole_class_map_by_cat

    def __len__(self):
        return len(self.list)
    
    def __getitem__(self, idx):
        X = self.list[idx]["text"]
        y_cat = self.list[idx]["category"]
        y = self.class_map[y_cat]
        return X, y
    
whole_dataset = TextDatasetByWhole()

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader, WeightedRandomSampler

whole_class_list = [y for _, y in whole_dataset]

train_indices, val_indices = train_test_split(list(range(len(whole_dataset))),
                                              test_size=0.2,
                                              stratify=whole_class_list)

whole_train_subset = Subset(whole_dataset, train_indices)
whole_val_subset = Subset(whole_dataset, val_indices)


print(f"[분할 결과]")
print(f"· 기존 학습 데이터: {len(whole_dataset)}개")
print(f"· 분할 후 학습 데이터: {len(train_indices)}개 ({len(train_indices) / len(whole_dataset) * 100:.2f}%)")
print(f"· 분할 후 검증 데이터: {len(val_indices)}개 ({len(val_indices) / len(whole_dataset) * 100:.2f}%)")

[분할 결과]
· 기존 학습 데이터: 38579개
· 분할 후 학습 데이터: 30863개 (80.00%)
· 분할 후 검증 데이터: 7716개 (20.00%)


In [194]:
labels = torch.tensor([whole_class_list[idx] for idx in train_indices])

class_counts = torch.bincount(labels)
class_weights = 1. / class_counts.float()
sampler_weights = class_weights[labels]

whole_sampler = WeightedRandomSampler(weights=sampler_weights,
                                      num_samples=len(sampler_weights),
                                      replacement=True)

print("[클래스별 가중치]")
for i, weight in enumerate(class_weights):
    print(f"· {i}({whole_class_map_by_num[i]}): {float(weight):.4f}")

[클래스별 가중치]
· 0(talk.politics.mideast): 0.0004
· 1(rec.autos): 0.0008
· 2(comp.sys.mac.hardware): 0.0010
· 3(alt.atheism): 0.0006
· 4(rec.sport.baseball): 0.0008
· 5(comp.os.ms-windows.misc): 0.0009
· 6(rec.sport.hockey): 0.0006
· 7(sci.crypt): 0.0005
· 8(sci.med): 0.0006
· 9(talk.politics.misc): 0.0007
· 10(rec.motorcycles): 0.0008
· 11(comp.windows.x): 0.0005
· 12(comp.graphics): 0.0006
· 13(comp.sys.ibm.pc.hardware): 0.0007
· 14(sci.electronics): 0.0008
· 15(talk.politics.guns): 0.0006
· 16(sci.space): 0.0005
· 17(soc.religion.christian): 0.0005
· 18(misc.forsale): 0.0009
· 19(talk.religion.misc): 0.0008


In [ ]:
WHOLE_TRAIN_DATALOADER = DataLoader(whole_train_subset, 
                                    batch_size=64, 
                                    shuffle=False, 
                                    num_workers=4,
                                    sampler=whole_sampler)

WHOLE_VAL_DATALOADER = DataLoader(whole_val_subset, 
                                  batch_size=64, 
                                  shuffle=False, 
                                  num_workers=4)

In [197]:
import spacy

ModuleNotFoundError: No module named 'spacy'